In [1]:
import pandas as pd
import re
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

ARQUIVO = "servidor__1_.csv"

print("=" * 55)
print("  ANÁLISE: Servidores da ANVISA")
print("=" * 55)

  ANÁLISE: Servidores da ANVISA


In [6]:
# ── 1. CARREGAR E LIMPAR ──────────────────────────────────────────────────────
print("\n[1/4] Carregando arquivo...")

df = pd.read_csv('servidor_ANVISA.csv', sep=";", encoding="utf-8-sig", dtype=str)
df.columns = df.columns.str.strip()
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"    Linhas brutas: {len(df):,}")


[1/4] Carregando arquivo...
    Linhas brutas: 1,897


In [7]:
# ── 2. REMOVER DUPLICATAS ─────────────────────────────────────────────────────
# Cada servidor aparece com 2 linhas: uma com cargo real e outra com "Sem informaç"
# Estratégia: por CPF, priorizar a linha com cargo real (não "Sem informaç")
print("\n[2/4] Removendo duplicatas...")

df["_tem_cargo"] = ~df["Cargo"].str.startswith("Sem informaç")

# Ordenar: linhas com cargo real primeiro
df = df.sort_values(["CPF", "_tem_cargo"], ascending=[True, False])

# Manter apenas a primeira ocorrência de cada CPF (a com cargo real, se existir)
df = df.drop_duplicates(subset="CPF", keep="first").drop(columns="_tem_cargo")
df = df.reset_index(drop=True)

total_servidores = len(df)
print(f"    Servidores únicos: {total_servidores:,}")


[2/4] Removendo duplicatas...
    Servidores únicos: 1,441


In [8]:
# ── 3. EXTRAIR UF DA UORG ─────────────────────────────────────────────────────
print("\n[3/4] Identificando UF por UORG...")

UFS_VALIDAS = {
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT",
    "PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"
}

def extrair_uf(uorg):
    if not isinstance(uorg, str):
        return "DF"
    match = re.search(r"-([A-Z]{2})\s*$", uorg.strip())
    if match:
        sigla = match.group(1)
        if sigla in UFS_VALIDAS:
            return sigla
    return "DF"

NOMES_ESTADOS = {
    "AC": "Acre", "AL": "Alagoas", "AM": "Amazonas", "AP": "Amapá",
    "BA": "Bahia", "CE": "Ceará", "DF": "Distrito Federal",
    "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão",
    "MG": "Minas Gerais", "MS": "Mato Grosso do Sul", "MT": "Mato Grosso",
    "PA": "Pará", "PB": "Paraíba", "PE": "Pernambuco", "PI": "Piauí",
    "PR": "Paraná", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
    "RO": "Rondônia", "RR": "Roraima", "RS": "Rio Grande do Sul",
    "SC": "Santa Catarina", "SE": "Sergipe", "SP": "São Paulo",
    "TO": "Tocantins"
}

df["UF"]     = df["UORG de Exercício (SIAPE)"].apply(extrair_uf)
df["Estado"] = df["UF"].map(NOMES_ESTADOS)


[3/4] Identificando UF por UORG...


In [9]:
# ── 4. AGREGAÇÕES ─────────────────────────────────────────────────────────────

# Por Estado
por_estado = (
    df.groupby(["UF", "Estado"])
    .size()
    .reset_index(name="Qtd_Servidores")
    .sort_values("Estado")
)
total_row_estado = pd.DataFrame([{
    "UF": "BR", "Estado": "TOTAL BRASIL", "Qtd_Servidores": total_servidores
}])
por_estado = pd.concat([total_row_estado, por_estado], ignore_index=True)

# Por Unidade (UORG) — mantém nome original
por_unidade = (
    df.groupby(["UORG de Exercício (SIAPE)", "UF", "Estado"])
    .size()
    .reset_index(name="Qtd_Servidores")
    .rename(columns={"UORG de Exercício (SIAPE)": "Unidade (UORG)"})
    .sort_values(["Estado", "Unidade (UORG)"])
)
total_row_unidade = pd.DataFrame([{
    "Unidade (UORG)": "TOTAL BRASIL", "UF": "BR",
    "Estado": "", "Qtd_Servidores": total_servidores
}])
por_unidade = pd.concat([total_row_unidade, por_unidade], ignore_index=True)

print(f"    Estados com servidores: {len(por_estado) - 1}")
print(f"    Unidades (UORGs) distintas: {len(por_unidade) - 1}")

    Estados com servidores: 26
    Unidades (UORGs) distintas: 187


In [10]:
# ── 5. EXPORTAR EXCEL ─────────────────────────────────────────────────────────
print("\n[4/4] Gerando relatórios Excel...")

COR_HEADER = "1F4E79"
COR_TOTAL  = "FFF2CC"
COR_PAR    = "D6E4F0"
COR_IMPAR  = "FFFFFF"

def formatar_excel(ws):
    borda = Border(bottom=Side(style="thin", color="BFBFBF"))

    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
        cell.fill      = PatternFill("solid", start_color=COR_HEADER)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = borda
    ws.row_dimensions[1].height = 32

    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        is_total = str(ws.cell(i, 1).value) in ("BR", "TOTAL BRASIL")
        for cell in row:
            cell.border = borda
            if is_total:
                cell.fill = PatternFill("solid", start_color=COR_TOTAL)
                cell.font = Font(bold=True, name="Arial", size=10)
            else:
                cor = COR_PAR if i % 2 == 0 else COR_IMPAR
                cell.fill = PatternFill("solid", start_color=cor)
                cell.font = Font(name="Arial", size=10)
            cell.alignment = Alignment(vertical="center")
            if isinstance(cell.value, (int, float)):
                cell.alignment     = Alignment(horizontal="right", vertical="center")
                cell.number_format = "#,##0"

    for col in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 55)

    ws.freeze_panes = "A2"

def adicionar_aba_sobre(wb, titulo):
    ws = wb.create_sheet("Sobre")
    meta = [
        ("Relatório",  titulo),
        ("Fonte",      "Portal da Transparência / GGPES - ANVISA"),
        ("Total",      f"{total_servidores} servidores únicos"),
        ("Nota dedup", "Servidores com 2 linhas (cargo real + 'Sem informaç') "
                       "foram deduplicados por CPF, mantendo a linha com cargo real."),
        ("Nota UF",    "UF extraída do sufixo '-XX' do nome da UORG. "
                       "UORGs sem UF identificável foram atribuídas ao DF."),
    ]
    for r, (chave, valor) in enumerate(meta, start=1):
        ws.cell(r, 1, chave).font = Font(bold=True, name="Arial", size=10)
        ws.cell(r, 2, valor).font = Font(name="Arial", size=10)
        ws.cell(r, 2).alignment   = Alignment(wrap_text=True)
    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 75

def salvar(df_out, nome_arquivo, titulo):
    df_out.to_excel(nome_arquivo, index=False, sheet_name="Dados")
    wb = load_workbook(nome_arquivo)
    formatar_excel(wb["Dados"])
    adicionar_aba_sobre(wb, titulo)
    wb.save(nome_arquivo)
    print(f"    ✅ {nome_arquivo}")

salvar(
    por_estado,
    "relatorio_anvisa_por_estado.xlsx",
    "Servidores da ANVISA por Estado"
)
salvar(
    por_unidade,
    "relatorio_anvisa_por_unidade.xlsx",
    "Servidores da ANVISA por Unidade (UORG)"
)

print(f"\n{'='*55}")
print(f"  CONCLUÍDO!")
print(f"  Total de servidores únicos: {total_servidores:,}")
print(f"  Arquivos gerados:")
print(f"    → relatorio_anvisa_por_estado.xlsx")
print(f"    → relatorio_anvisa_por_unidade.xlsx")
print(f"{'='*55}")


[4/4] Gerando relatórios Excel...
    ✅ relatorio_anvisa_por_estado.xlsx
    ✅ relatorio_anvisa_por_unidade.xlsx

  CONCLUÍDO!
  Total de servidores únicos: 1,441
  Arquivos gerados:
    → relatorio_anvisa_por_estado.xlsx
    → relatorio_anvisa_por_unidade.xlsx
